In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# from catboost import CatBoostClassifier


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
file_path = os.path.join(path, 'Q3_data.csv')
df_credit = pd.read_csv(file_path)

In [ ]:
# Task 2: Write your code here:
df_credit.head() #In target, 1 Default, 0 No Default

In [ ]:
# Task 3: Write your code here:
df_credit.info()

In [ ]:
# Task 4: Write your code here:
df_credit.describe() #Numerical data needs scaling, some values are missing

In [ ]:
# Task 1: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_credit) # I think I should drop D_142, 16858 missing values that's a lot, but don't rush this, keep going
# I need to decide now, this is a sensitive task and I don't wanna make a model that predicts something this sensitive using incomplete data, I think dropping dropna() is the way to go

num_cols = df_credit.select_dtypes(include=["float", "int"]).columns

df_clean = df_credit[num_cols].fillna(0)
df_clean.shape


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)
# No duplicates

In [ ]:
# Task 3: Write your code here:

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
categorical_cols.value_counts()

# No categorical variables

In [ ]:
# Task 4: Write your code here:

features = df_clean.columns.drop('Target')
print(features)
X = df_clean[features]
print(X.head())
print('========================')
y = df_clean['Target']
print(y.head())

# I wanna split the data before scaling, to prevent data leakage, best practices and stuff, so I'll jump some steps if that's ok whoever you are :)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # I will do fit_transform here, and I'll only transform and X_test so the statistcs used to scale the data won't let the model cheat
X_test_scaled = scaler.transform(X_test) # Only transform here

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()



In [ ]:
# Task 5: Write your code here:

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, 'Target')

# Data is heavily imbalanced, stratify and use F1 score

In [ ]:
# Task 1: Write your code here:
# Already done

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# from catboost import CatBoostClassifier #Since the import isn't working, I'll create other models, please forgive me catboost isn't importing for some reason

In [ ]:
# Task 2,3,4,5: Write your code here:
sklearn_models = {
  "K-Nearest Neighbors": KNeighborsClassifier(
      n_neighbors=3,  # Number of neighbors to consider
  ),
  "Support Vector Machine": SVC(
      kernel='rbf',  # 'linear', 'poly', 'rbf', 'sigmoid'
      C=0.75  # Regularization parameter
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=3  # Maximum depth of tree (prevents overfitting)
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=300,  # Number of boosting rounds
      max_depth=5,
      learning_rate=0.05 # Step size shrinkage
  ),
}



In [ ]:
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items(): # .items() gives you a pair of the key and its value (value could be a dict to so this is really useful)
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
  print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Calculate the majority class baseline
majority_class = y.value_counts().idxmax() # This returns the label of the majority class
baseline_pred = [majority_class] * len(y) # This creates a list with 1 repeated to the length of y, [1, 1, 1, 1, 1, ...]

# Evaluate the baseline
baseline_accuracy = accuracy_score(y, baseline_pred)
baseline_precision = precision_score(y, baseline_pred, average='weighted', zero_division=0) # zero_division=0 means that if there was zero division just returns 0, to avoid errors
baseline_recall = recall_score(y, baseline_pred, average='weighted', zero_division=0)
baseline_f1 = f1_score(y, baseline_pred, average='weighted', zero_division=0)

print(f"Baseline Accuracy (majority class): {baseline_accuracy:.4f}")
print(f"Baseline Precision: {baseline_precision:.4f}")
print(f"Baseline Recall: {baseline_recall:.4f}")
print(f"Baseline F1-Score: {baseline_f1:.4f}")

In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = # I'm running out of time and I can't wait for the models to get trained, but the golden feature will be the one with the highest bar in the graphs

In [ ]:
# Task Bonus: Write your code here: